# GRM shard accumulate -- parallel

Same `accumulate`/`merge` step as `grm_shard_processing.ipynb`, but runs
the per-shard `accumulate` calls concurrently instead of one at a time --
each `(phenotype, transform, covariate_set, shard)` call is fully
independent (its own subprocess, no shared state), so this is
embarrassingly parallel.

**Machine**: request `n1-highcpu-16` (16 vCPU / ~14.4GB RAM), not
standard/highmem -- `accumulate` doesn't scan the genotype panel (that
was plink's fixed cost, not this tool's), so it shouldn't need much
memory per process. Set `N_CONCURRENT = 8` below (half the vCPUs) as a
starting point -- n1 machines are 2-way SMT, and we got burned before
assuming logical-core count = free parallelism with plink's actual
compute-heavy pass. If a first run at 8 shows low CPU utilization and
good headroom, try bumping toward 16 next time; don't jump straight
there unbenchmarked.

Every produced `.acc.tsv`/`.full.tsv` is uploaded to the bucket
(`CROSSPRODUCTS_BUCKET_GS`) as soon as it's written -- not batched at the
end -- so an interrupted run doesn't lose already-finished work, and a
rerun (local or on a fresh VM) can skip anything already done, in
either place.

In [ ]:
import glob
import hashlib
import os
import re
import subprocess
from datetime import datetime, timezone

import pandas as pd

REPO_DIR = os.path.expanduser("~/repos/aou-covariance")
TOOL_DIR = f"{REPO_DIR}/GRM-pairs/grm_bin_sharded"
TOOL_BIN = f"{TOOL_DIR}/grm_shard_tool"
BINS_FILE = f"{REPO_DIR}/GRM-pairs/full_grm_bin/bins.txt"

subprocess.run(["make"], cwd=TOOL_DIR, check=True)
assert os.path.isfile(TOOL_BIN), "missing tool binary: " + TOOL_BIN
assert os.path.isfile(BINS_FILE), "missing bins file: " + BINS_FILE

WORKSPACE_BUCKET = os.path.expanduser(
    "~/workspace/Data from All of Us Controlled Tier /shared-env-pilot"
)
WORKSPACE_BUCKET_GS = "gs://shared-env-pilot-wb-fulgent-almond-9474"
BUCKET_DIR = f"{WORKSPACE_BUCKET}/data/03_grm_shards"
CROSSPRODUCTS_BUCKET_GS = f"{WORKSPACE_BUCKET_GS}/data/04_process_shards/crossproducts"

WORK_DIR = os.path.expanduser("~/grm_pheno_cov")
os.makedirs(WORK_DIR, exist_ok=True)

N_SHARDS = 16
NBLOCKS = 50
SEED = 1
N_CONCURRENT = 8   # see machine note above -- n1-highcpu-16, start at half the vCPUs

SHARDS_DIR = f"{BUCKET_DIR}/shards"
_canonical_id = f"{SHARDS_DIR}/genome_wide.grm.id"
if os.path.isfile(_canonical_id):
    GRM_ID = _canonical_id
else:
    _any_id = sorted(glob.glob(f"{SHARDS_DIR}/grm_shard_*_of_{N_SHARDS}.grm.id"))
    assert _any_id, f"no .grm.id file found under {SHARDS_DIR}"
    GRM_ID = _any_id[0]

available_shards = sorted(glob.glob(f"{SHARDS_DIR}/grm_shard_*_of_{N_SHARDS}.grm.bin.*"))
print(f"GRM_ID = {GRM_ID}")
print(f"{len(available_shards)}/{N_SHARDS} shards available so far")

## FID alignment

Same as `grm_shard_processing.ipynb` -- rewrites each phenotype file's
`FID` from the real `.grm.id` (keyed on `IID`), with a provenance header.

In [ ]:
grm_ids = pd.read_csv(GRM_ID, sep=r"\s+", header=None, names=["FID", "IID"], dtype=str)
id_map = dict(zip(grm_ids["IID"], grm_ids["FID"]))


def build_aligned_pheno(pheno_path, out_path):
    pheno = pd.read_csv(pheno_path, sep=r"\s+", dtype={"FID": str, "IID": str})
    n_source = len(pheno)
    pheno["FID"] = pheno["IID"].map(id_map)
    unmatched = int(pheno["FID"].isna().sum())
    if unmatched:
        pheno = pheno.dropna(subset=["FID"])
    n_matched = len(pheno)

    id_fingerprint = hashlib.sha256(
        "\n".join(f"{fid} {iid}" for fid, iid in sorted(zip(pheno["FID"], pheno["IID"])))
        .encode()
    ).hexdigest()

    with open(out_path, "w") as f:
        f.write(f"# source={pheno_path}\n")
        f.write(f"# generated={datetime.now(timezone.utc).isoformat()}\n")
        f.write(f"# grm_id={GRM_ID}\n")
        f.write(f"# n_source_rows={n_source}\n")
        f.write(f"# n_matched_grm={n_matched}\n")
        f.write(f"# n_dropped_unmatched={unmatched}\n")
        f.write(f"# id_set_sha256={id_fingerprint}\n")
    pheno.to_csv(out_path, sep=" ", index=False, columns=["FID", "IID", "Y"], mode="a")
    return out_path

## Phenotypes

Same systematic discovery as `grm_shard_processing.ipynb`. Narrow
`TARGET_PHENOTYPES` to a subset if you don't want to queue every
phenotype/transform/covariate-set combo -- leave it as `None` to run
everything found.

In [ ]:
PHENO_DIR = os.path.expanduser(
    "~/workspace/Data from All of Us Controlled Tier /shared-env-pilot/data/02_phenotype/residualized"
)
TARGET_PHENOTYPES = None   # e.g. {"height", "bmi", "systolic_bp", "heart_rate"}, or None for all

combos = []
for fname in sorted(os.listdir(PHENO_DIR)):
    if not fname.endswith(".pheno"):
        continue
    stem = fname[: -len(".pheno")]
    parts = stem.rsplit("__", 2)
    if len(parts) != 3:
        continue
    phenotype_name, transform, covariate_set = parts
    if TARGET_PHENOTYPES is not None and phenotype_name not in TARGET_PHENOTYPES:
        continue
    combos.append((phenotype_name, transform, covariate_set, fname))

print(f"{len(combos)} combos queued, {len(available_shards)} shards each = "
      f"{len(combos) * len(available_shards)} accumulate calls")

## Skip-if-exists (local or bucket)

Lists `CROSSPRODUCTS_BUCKET_GS` once up front rather than doing a
`gcloud storage ls` per file (would be one network round-trip per shard
otherwise -- thousands of them at full scale).

In [ ]:
def list_bucket_files(bucket_prefix):
    result = subprocess.run(["gcloud", "storage", "ls", bucket_prefix],
                             capture_output=True, text=True)
    if result.returncode != 0:
        return set()
    return {line.strip().rsplit("/", 1)[-1] for line in result.stdout.splitlines() if line.strip()}


_bucket_files = list_bucket_files(f"{CROSSPRODUCTS_BUCKET_GS}/*")
print(f"{len(_bucket_files)} files already in {CROSSPRODUCTS_BUCKET_GS}")


def file_done(local_path):
    if os.path.isfile(local_path) and os.path.getsize(local_path) > 0:
        return True
    return os.path.basename(local_path) in _bucket_files

## Parallel accumulate

`process_shard` is a plain module-level function so `ProcessPoolExecutor`
can fork worker processes for it (Jupyter's default start method on
Linux is `fork`, which inherits the parent's already-defined globals --
no re-import needed, unlike `spawn`).

In [ ]:
from concurrent.futures import ProcessPoolExecutor, as_completed


def process_shard(tag, shard_bin, k, aligned_pheno):
    acc_out = f"{WORK_DIR}/{tag}_shard{k}.acc.tsv"
    if file_done(acc_out):
        return acc_out
    subprocess.run(
        [TOOL_BIN, "accumulate",
         "--grm-id", GRM_ID, "--shard", shard_bin,
         "--parallel", str(k), str(N_SHARDS),
         "--pheno", aligned_pheno, "--bins", BINS_FILE,
         "--nblocks", str(NBLOCKS), "--seed", str(SEED),
         "--out", acc_out],
        check=True,
    )
    subprocess.run(["gcloud", "storage", "cp", acc_out, f"{CROSSPRODUCTS_BUCKET_GS}/"], check=True)
    return acc_out


tasks = []
for phenotype_name, transform, covariate_set, fname in combos:
    tag = f"{phenotype_name}__{transform}__{covariate_set}"
    aligned = build_aligned_pheno(f"{PHENO_DIR}/{fname}", f"{WORK_DIR}/{tag}_aligned.pheno")
    for shard_bin in available_shards:
        m = re.search(r"grm_shard_(\d+)_of_\d+\.grm\.bin\.\d+$", shard_bin)
        k = int(m.group(1))
        tasks.append((tag, shard_bin, k, aligned))

print(f"{len(tasks)} shard-accumulate tasks queued, N_CONCURRENT={N_CONCURRENT}")

completed = 0
with ProcessPoolExecutor(max_workers=N_CONCURRENT) as executor:
    futures = {executor.submit(process_shard, *t): t for t in tasks}
    for future in as_completed(futures):
        future.result()   # raises immediately if a shard's accumulate call failed
        completed += 1
        if completed % 20 == 0 or completed == len(tasks):
            print(f"  {completed}/{len(tasks)} done")

print("all accumulate tasks finished")

## Merge + upload

Sequential (cheap relative to accumulate) -- one `merge` per combo,
uploading the resulting `.full.tsv` alongside the accumulator files.

In [ ]:
merged_paths = []
for phenotype_name, transform, covariate_set, fname in combos:
    tag = f"{phenotype_name}__{transform}__{covariate_set}"
    acc_files = sorted(glob.glob(f"{WORK_DIR}/{tag}_shard*.acc.tsv"))
    assert len(acc_files) == len(available_shards), (
        f"{tag}: expected {len(available_shards)} acc files, found {len(acc_files)}"
    )
    acc_list = f"{WORK_DIR}/{tag}_acc_list.txt"
    with open(acc_list, "w") as f:
        f.write("\n".join(acc_files) + "\n")

    out_prefix = f"{WORK_DIR}/{tag}_merged"
    subprocess.run(
        [TOOL_BIN, "merge", "--acc-list", acc_list, "--bins", BINS_FILE,
         "--nblocks", str(NBLOCKS), "--out-prefix", out_prefix],
        check=True,
    )
    full_tsv = f"{out_prefix}.full.tsv"
    subprocess.run(["gcloud", "storage", "cp", full_tsv, f"{CROSSPRODUCTS_BUCKET_GS}/"], check=True)
    merged_paths.append(full_tsv)
    print(f"  merged + uploaded {tag}")

print(f"done -- {len(merged_paths)} combos merged and uploaded to {CROSSPRODUCTS_BUCKET_GS}")